In [1]:
import pandas as pd

In [2]:
tabula = pd.read_excel('/workspaces/CUBES/exp/jack/Data/UK_Data/Tabula_Condensed.xlsx')


In [3]:
tab = tabula[tabula['Code_BuildingVariant'].str.contains('ReEx') & tabula['Code_StatusDataset'].str.contains('Typology')]
tab = tab.reset_index(drop=True)

In [4]:
tab_gb = tab[tab['Code_Country'] == 'GB']

In [5]:
columns_wanted = [
                    'Code_BuildingVariant',
                    'A_Floor_1',
                    'A_Floor_2',
                    'A_Wall_1',
                    'A_Wall_2',
                    'A_Wall_3',
                    'A_Window_1',
                    'A_Window_2',
                    'A_Window_East',
                    'A_Window_South',
                    'A_Window_West',
                    'A_Window_North',
                    'A_Estim_Roof',
                    'A_Roof_1',
                    'A_Roof_2',
                    'A_Door_1',
                    'n_Storey',
                    'n_Apartment',
                    'Code_AttachedNeighbours',
                    'Code_AtticCond',
                    'h_Ceiling',
                    'delta_U_ThermalBridging_Original',
                    'n_air_infiltration',
                    'U_Roof_1',
                    'U_Roof_2',
                    'U_Wall_1',
                    'U_Wall_2',
                    'U_Wall_3',
                    'U_Floor_1',
                    'U_Floor_2',
                    'U_Window_1',
                    'U_Window_2',
                    'U_Door_1',
                    'g_gl_n_Window_1',
                    'g_gl_n_Window_2',
                    'd_Insulation_Roof_1',
                    'd_Insulation_Roof_2',
                    'd_Insulation_Wall_1',
                    'd_Insulation_Wall_2',
                    'd_Insulation_Wall_3',
                    'd_Insulation_Floor_1',
                    'd_Insulation_Floor_2']

In [6]:
tab_gb = tab_gb[columns_wanted]

In [14]:
constructions = pd.read_csv('/workspaces/CUBES/exp/jack/Data/tabula/tabula_gb_constructions.csv')
map_constructions = pd.read_csv('/workspaces/CUBES/exp/jack/Data/cubes/maps/map_tabula_constructions_to_cubes.csv')
num_buildings = pd.read_csv('/workspaces/CUBES/exp/jack/Data/cubes/gb_num_buildings.csv')
ambience = pd.read_excel('/workspaces/CUBES/exp/jack/Data/ambience/AmBIENCe_Geometry_Constructions.xlsx')

In [15]:
gb_ambience = constructions.merge(num_buildings)
gb_ambience = gb_ambience.merge(tab_gb)

In [16]:
for index in gb_ambience.index:
    split_code = gb_ambience.loc[index]['Code_BuildingVariant'].split('.')
    gb_ambience.loc[index, 'Code_BuildingVariant'] = split_code[0]+'-'+split_code[2]+'-'+split_code[3]


elements = ['Roof','Wall','Floor']
for index, entry in gb_ambience.iterrows():
    for element in elements:
        tabula_construction = gb_ambience.loc[index][element]
        uwe_construction = map_constructions[map_constructions['TABULA_Nomenclature'] == tabula_construction]['CUBES_Nomenclature'].values[0]
        gb_ambience.loc[index, element] = uwe_construction


gb_ambience['Floor_Area'] = gb_ambience['A_Floor_1']+gb_ambience['A_Floor_2']
gb_ambience['Wall_Area'] = gb_ambience['A_Wall_1']+gb_ambience['A_Wall_2']+gb_ambience['A_Wall_3']
gb_ambience['Window_Area'] = gb_ambience['A_Window_1']+gb_ambience['A_Window_2']
gb_ambience['Roof_Area'] = gb_ambience['A_Roof_1']+gb_ambience['A_Roof_2']
gb_ambience['Number Of Buildings'] = gb_ambience['Number Of Buildings']*1000000 

Renaming = {'Code_BuildingVariant':'REFERENCE BUILDING CODE',
'Floor_Area': "REFERENCE BUILDING GROUND FLOOR AREA (m2)",
'Wall_Area':"REFERENCE BUILDING WALL AREA (m2)",
'Window_Area':"REFERENCE BUILDING WINDOW AREA (m2)",
'Roof_Area':"REFERENCE BUILDING ROOF AREA (m2)",
'n_Storey':"NUMBER OF REFERENCE BUILDING STOREYS",
'Number Of Buildings':"BUILDING STOCK SEGMENT NUMBER OF BUILDINGS",
'Floor': 'FLOOR', 
'Roof': 'ROOF',
"Wall": 'WALL', 
'Window': 'WINDOW', 
'Door':'DOOR'}

gb_ambience.rename(columns=Renaming,inplace=True)


In [17]:
new_df = pd.concat([ambience,gb_ambience])

In [19]:
gb_ambience.to_csv('/workspaces/CUBES/exp/jack/Data/cubes/gb_Geometry_Construction.csv')